<a href="https://colab.research.google.com/github/MarkowitzMx/Programacion-para-analitica-descriptiva-y-predictiva-2026/blob/main/Actividad_Calificada_Particion_Remuestreo_Arbol_274690.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Actividad Calificada: Partición y Remuestreo con Árbol de Decisión

**Programación para Analítica Descriptiva y Predictiva — MIAAD, UACJ**


**Nombre: Cristóbal Lemus Rendón **

**Matricula: 274690 **

**Ficha técnica**

| | |
|---|---|
| **Modalidad** | Individual |
| **Entorno** | Google Colab |
| **Dataset** | Titanic (pasajeros del RMS Titanic, clasificación binaria) |
| **Modelo** | Árbol de Decisión (`DecisionTreeClassifier`) |
| **Puntaje total** | 100 puntos |
| **Fecha de entrega** | 21 Octubre 2026 |

**Objetivo**

Aplicar partición hold-out estratificada y dos esquemas de validación cruzada (`StratifiedKFold` y `RepeatedStratifiedKFold`) para estimar el desempeño de un Árbol de Decisión, y justificar, con base en el sesgo y la varianza de la estimación, qué método reportarías como resultado final.

**Dataset**

El dataset contiene 891 pasajeros del Titanic, con la variable `survived` (0 = no sobrevivió, 1 = sobrevivió) como variable objetivo. Tiene valores ausentes genuinos en `age` (177 casos) y `embarked` (2 casos) — no son valores agregados artificialmente para esta actividad.

**Instrucciones generales**

- Completa únicamente las celdas marcadas con `# TODO`. No modifiques las celdas de preparación de datos ni los hiperparámetros ya indicados (`max_depth=4`, `random_state=0`), para que tus resultados sean comparables con los de tus compañeros.
- Cada actividad tiene una parte de código y una pregunta de interpretación en una celda de texto (Markdown). Responde directamente debajo de **Tu respuesta:**.
- No se califican fragmentos de código comentados o dejados sin completar (`# TODO` sin resolver).
- Antes de entregar, ejecuta todo el notebook de principio a fin (`Entorno de ejecución > Reiniciar y ejecutar todo` en Colab) y verifica que no queden errores.

**Entrega**

Sube a Moodle un archivo `.txt` con el enlace a tu notebook de Colab o GitHub (con permisos de lectura habilitados). La subida del archivo `.ipynb` es opcional, pero recomendada como respaldo.


## Preparación de los datos

Esta sección se proporciona completa: no es parte de la evaluación. Carga el dataset, selecciona las variables predictoras y aplica un tratamiento mínimo a los valores ausentes y a las variables categóricas, para que puedas concentrarte en la partición y el remuestreo.


In [5]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, RepeatedStratifiedKFold, cross_validate
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.precision", 3)

url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
df = pd.read_csv(url)

print(f"Observaciones: {df.shape[0]}, columnas: {df.shape[1]}")
print("Valores ausentes por columna (solo las que tienen alguno):")
print(df.isna().sum()[df.isna().sum() > 0])

# Variables predictoras: se dejan fuera las que son redundantes con 'survived'
# (alive, who, adult_male), las de muy alta cardinalidad (deck, con 688 ausentes)
# o duplicadas en otra forma (class es la versión en texto de pclass).
variables = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]
X = df[variables].copy()
y = df["survived"].copy()

# Tratamiento mínimo, dado: imputar age con la mediana y embarked con la moda,
# y codificar las dos variables categóricas como enteros.
X["age"] = X["age"].fillna(X["age"].median())
X["embarked"] = X["embarked"].fillna(X["embarked"].mode()[0])
X["sex"] = X["sex"].map({"male": 0, "female": 1})
X["embarked"] = X["embarked"].map({"S": 0, "C": 1, "Q": 2})

print(f"\nVariables predictoras: {list(X.columns)}")
print(f"Distribución de la variable objetivo (survived): {dict(y.value_counts().sort_index())}")


Observaciones: 891, columnas: 15
Valores ausentes por columna (solo las que tienen alguno):
age            177
embarked         2
deck           688
embark_town      2
dtype: int64

Variables predictoras: ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
Distribución de la variable objetivo (survived): {0: np.int64(549), 1: np.int64(342)}


---
## Actividad 1 — Hold-out estratificado (20 pts)

**1.1 (15 pts).** Completa la partición hold-out: reserva el 20% de las observaciones para prueba, con estratificación por `y` y `random_state=42`. Ajusta un `DecisionTreeClassifier` con `max_depth=4` y `random_state=0` usando el conjunto de entrenamiento, y calcula su accuracy sobre el conjunto de prueba.


In [6]:
# Actividad 1.1: partición hold-out y ajuste del modelo.
X_train, X_test, y_train, y_test = train_test_split(  # divide los datos en entrenamiento y prueba y devuelve 4 objetos
    X,                # matriz de variables predictoras que se va a dividir
    y,                # variable objetivo (survived) que se divide con las mismas filas que X
    test_size=0.2,    # reserva el 20% de las observaciones para prueba (80% queda para entrenar)
    stratify=y,       # mantiene en ambos conjuntos la misma proporción de sobrevivientes que en y (~38%)
    random_state=42   # semilla fija: la partición es reproducible y comparable con la de los compañeros
)

arbol = DecisionTreeClassifier(max_depth=4, random_state=0)  # crea el árbol limitado a profundidad 4 (evita sobreajuste); semilla 0 para desempates reproducibles
arbol.fit(X_train, y_train)  # entrena el árbol usando solo el conjunto de entrenamiento

acc_holdout = accuracy_score(y_test, arbol.predict(X_test))  # predice sobre prueba y compara con las etiquetas reales: proporción de aciertos

print(f"Observaciones de entrenamiento: {X_train.shape[0]}")  # imprime cuántas filas tiene el conjunto de entrenamiento (712)
print(f"Observaciones de prueba: {X_test.shape[0]}")          # imprime cuántas filas tiene el conjunto de prueba (179)
print(f"Accuracy hold-out (prueba): {acc_holdout:.4f}")       # imprime el accuracy en prueba con 4 decimales

Observaciones de entrenamiento: 712
Observaciones de prueba: 179
Accuracy hold-out (prueba): 0.7877


**Interpretación 1.1.** El conjunto de entrenamiento tiene 712 observaciones y el de prueba 179 (80/20). El árbol de profundidad 4 acierta en ~**78.8%** de los pasajeros de prueba (accuracy = 0.788): es decir, clasifica correctamente casi 4 de cada 5 pasajeros que no vio durante el entrenamiento. Es una única estimación, por lo que su representatividad se discute en la Actividad 2.


**1.2 (5 pts).** El Titanic tiene una proporción de supervivencia desbalanceada (aproximadamente 38% sobrevivió, 62% no). ¿Qué problema evita la estratificación (`stratify=y`) en este caso específico, y qué podría pasar con la partición si no se usara?

**Tu respuesta:**

En el Titanic sobrevivió 38% de los pasajeros y no sobrevivió 62%, es decir, las clases están desbalanceadas. La estratificación (`stratify=y`) garantiza que tanto el conjunto de entrenamiento como el de prueba conserven esa misma proporción (~38/62) de la variable objetivo.

Si no se usara, la partición sería puramente aleatoria y, por azar, el conjunto de prueba podría quedar con una proporción de sobrevivientes distinta a la real (por ejemplo 30% o 45%). Con solo 179 observaciones de prueba esa fluctuación es considerable. Esto haría que el accuracy dependiera de qué tan "fácil" o "difícil" quedó la muestra de prueba y no solo del modelo, y además el entrenamiento podría ver una proporción de clases distinta a la de la población. La estratificación reduce esa fuente de variabilidad y hace que la estimación sea más estable y representativa.

---
## Actividad 2 — StratifiedKFold (25 pts)

**2.1 (15 pts).** Completa la validación cruzada con `StratifiedKFold`: 5 folds, `shuffle=True` y `random_state=42`. Usa `cross_validate` con un `DecisionTreeClassifier(max_depth=4, random_state=0)` nuevo (mismos hiperparámetros que en la Actividad 1) y reporta la media y la desviación estándar (`ddof=1`) del accuracy sobre los 5 folds.


In [7]:
# Actividad 2.1: validación cruzada con StratifiedKFold.
cv_kfold = StratifiedKFold(  # define el esquema de validación cruzada estratificada
    n_splits=5,       # divide los datos en 5 folds: cada uno se usa una vez como prueba y 4 veces como entrenamiento
    shuffle=True,     # mezcla los datos antes de formar los folds (evita depender del orden original)
    random_state=42   # semilla fija para que el mezclado, y por tanto los folds, sean reproducibles
)

arbol_kfold = DecisionTreeClassifier(max_depth=4, random_state=0)  # árbol nuevo, con los mismos hiperparámetros que en la Actividad 1
resultados_kfold = cross_validate(  # entrena y evalúa el modelo en cada fold; devuelve un diccionario de resultados
    arbol_kfold,          # modelo a evaluar (cross_validate lo clona y reentrena en cada fold)
    X,                    # todas las variables predictoras (la función hace la partición internamente)
    y,                    # variable objetivo
    cv=cv_kfold,          # esquema de folds definido arriba
    scoring="accuracy"    # métrica a calcular en cada fold de prueba
)

scores_kfold = resultados_kfold["test_score"]  # extrae el arreglo con el accuracy de cada uno de los 5 folds

print(f"Accuracy por fold: {scores_kfold}")                                  # imprime los 5 accuracies individuales
print(f"Media del accuracy: {scores_kfold.mean():.4f}")                      # imprime el promedio de los 5 folds
print(f"Desviación estándar (ddof=1): {scores_kfold.std(ddof=1):.4f}")       # desviación estándar muestral (divide entre n-1, no entre n)

Accuracy por fold: [0.832 0.826 0.792 0.809 0.837]
Media del accuracy: 0.8193
Desviación estándar (ddof=1): 0.0185


**2.2 (10 pts).** Compara la media del accuracy de `StratifiedKFold` con el accuracy único que obtuviste en la Actividad 1. ¿Por qué un solo valor de hold-out no es suficiente para saber si ese resultado es representativo del desempeño real del modelo?

**Tu respuesta:**

El accuracy del hold-out fue **0.788** y la media de `StratifiedKFold` fue **0.819** (desviación estándar 0.019). Son valores distintos: la diferencia (~3 puntos porcentuales) es mayor que la desviación estándar entre folds, lo que muestra que el resultado del hold-out depende mucho de qué observaciones cayeron en prueba.

Un solo valor de hold-out proviene de una única partición, con apenas 179 observaciones de prueba, así que no sabemos cuánto cambiaría si la partición fuera otra; no trae ninguna medida de su propia variabilidad. Pudo ser una partición "difícil" (subestima el desempeño) o "fácil" (lo sobreestima), y con un solo número no hay forma de distinguirlo. Además, el modelo se entrena con solo el 80% de los datos.

La validación cruzada k-fold evalúa el modelo en 5 particiones distintas, de modo que cada observación se usa una vez para prueba. Promediar los folds reduce la varianza de la estimación, y la desviación estándar entre folds da una idea de qué tan estable es el desempeño. Así sabemos si el resultado es representativo.

---
## Actividad 3 — RepeatedStratifiedKFold (25 pts)

**3.1 (15 pts).** Completa la validación cruzada repetida: 5 folds, 5 repeticiones y `random_state=42`. Usa `cross_validate` con el mismo modelo (`DecisionTreeClassifier(max_depth=4, random_state=0)`) y reporta la media y la desviación estándar (`ddof=1`) sobre las 25 estimaciones resultantes.


In [8]:
# Actividad 3.1: validación cruzada repetida.
cv_repetido = RepeatedStratifiedKFold(  # repite la validación cruzada estratificada varias veces con folds distintos
    n_splits=5,       # 5 folds en cada repetición
    n_repeats=5,      # se repite el proceso 5 veces, cada vez con una partición en folds diferente
    random_state=42   # semilla fija para reproducir todas las repeticiones
)

arbol_repetido = DecisionTreeClassifier(max_depth=4, random_state=0)  # mismo modelo y mismos hiperparámetros que antes
resultados_repetido = cross_validate(  # entrena y evalúa el modelo en cada uno de los 5 x 5 = 25 folds
    arbol_repetido,       # modelo a evaluar
    X,                    # variables predictoras
    y,                    # variable objetivo
    cv=cv_repetido,       # esquema repetido definido arriba
    scoring="accuracy"    # métrica a calcular en cada fold de prueba
)

scores_repetido = resultados_repetido["test_score"]  # extrae el arreglo con las 25 estimaciones de accuracy

print(f"Número de estimaciones: {len(scores_repetido)}")                         # imprime cuántas estimaciones hay (debe ser 25)
print(f"Media del accuracy: {scores_repetido.mean():.4f}")                       # imprime el promedio de las 25 estimaciones
print(f"Desviación estándar (ddof=1): {scores_repetido.std(ddof=1):.4f}")        # desviación estándar muestral de las 25 estimaciones

Número de estimaciones: 25
Media del accuracy: 0.8088
Desviación estándar (ddof=1): 0.0217


**3.2 (10 pts).** Compara la desviación estándar de `RepeatedStratifiedKFold` con la de `StratifiedKFold` (Actividad 2). ¿Qué fuente de variabilidad adicional captura la versión repetida que una sola corrida de `StratifiedKFold` no puede capturar?

**Tu respuesta:**

La desviación estándar de `RepeatedStratifiedKFold` fue **0.022**, ligeramente mayor que la de `StratifiedKFold` (**0.019**). Esto no significa que el método sea "peor": captura más variabilidad real que una sola corrida no puede ver.

Una sola corrida de `StratifiedKFold` usa una única configuración de folds (una sola forma de repartir las observaciones en 5 grupos). Su desviación estándar solo refleja las diferencias entre esos 5 folds. La versión repetida vuelve a barajar y rehacer la partición en folds en cada una de las 5 repeticiones, así que además incorpora la variabilidad debida a **cómo se reparten las observaciones en los folds**: distintas configuraciones de folds dan estimaciones algo distintas. Con 25 estimaciones se obtiene una imagen más completa de la dispersión.

Además, la media sobre 25 estimaciones es más estable (menor varianza de la media) que la de 5, porque depende menos de una partición particular.

---
## Actividad 4 — Comparación y justificación (15 pts)

**4.1 (10 pts).** Completa la tabla comparativa con los tres resultados obtenidos. Para hold-out, al ser una sola estimación, no hay desviación estándar.


In [9]:
# Actividad 4.1: tabla comparativa con los tres métodos.
resumen = pd.DataFrame({  # construye un DataFrame con una fila por método
    "método": ["Hold-out", "StratifiedKFold", "RepeatedStratifiedKFold"],  # nombres de los tres métodos
    "n_estimaciones": [1, len(scores_kfold), len(scores_repetido)],  # número de estimaciones: 1, 5 y 25
    "media_accuracy": [acc_holdout, scores_kfold.mean(), scores_repetido.mean()],  # media de accuracy de cada método
    "desv_est": [np.nan, scores_kfold.std(ddof=1), scores_repetido.std(ddof=1)],  # desviación estándar (ddof=1); np.nan en hold-out porque solo hay 1 valor
})
resumen  # muestra la tabla como salida de la celda

,método,n_estimaciones,media_accuracy,desv_est
0,Hold-out,1,0.788,NaN
1,StratifiedKFold,5,0.819,0.019
2,RepeatedStratifiedKFold,25,0.809,0.022


**4.2 (5 pts).** Si tuvieras que reportar un solo número como el desempeño esperado de este Árbol de Decisión en el dataset del Titanic, ¿cuál de los tres métodos usarías y por qué? Justifica tu respuesta en términos de sesgo y varianza de la estimación, no solo del valor de accuracy obtenido.

**Tu respuesta:**

Reportaría el resultado de **`RepeatedStratifiedKFold`** (media ≈ **0.809**, con desviación estándar ≈ 0.022 sobre 25 estimaciones).

**Varianza:** el hold-out es una sola estimación y tiene la mayor varianza: cambiar el `random_state` de la partición cambiaría el número. `StratifiedKFold` promedia 5 estimaciones, pero depende de una única forma de armar los folds. `RepeatedStratifiedKFold` promedia 25 estimaciones sobre 5 configuraciones distintas de folds, por lo que la media tiene la menor varianza de los tres y la desviación estándar describe mejor la incertidumbre del resultado.

**Sesgo:** en el hold-out el modelo se entrena con el 80% de los datos y se evalúa en apenas 179 casos; en k-fold de 5 se entrena también con el 80% en cada fold, pero se usan todas las observaciones para evaluar. Ambos esquemas k-fold tienen un sesgo pequeño y similar entre sí (el modelo ve el 80% de los datos), así que la ventaja de la versión repetida está sobre todo en la reducción de la varianza, no del sesgo.

La elección no se basa en que dé el accuracy más alto (de hecho `StratifiedKFold` dio 0.819, más alto que 0.809), sino en que es la estimación más estable y confiable del desempeño esperado. Su costo es entrenar el modelo 25 veces, algo barato para un árbol de profundidad 4 con 891 filas.

---
## Reflexión final (15 pts)

Responde en 6 a 10 líneas: ¿qué papel juega la elección del método de partición o remuestreo cuando se evalúa un modelo de clasificación, y qué factores del problema (tamaño del dataset, balance de clases, costo de entrenamiento) tomarías en cuenta para elegir uno sobre otro en un caso distinto al de esta actividad?

**Tu respuesta:**

La elección del método de partición o remuestreo determina qué tan confiable es el desempeño que reportamos: el mismo árbol obtuvo 0.788 con hold-out, 0.819 con StratifiedKFold y 0.809 con RepeatedStratifiedKFold, así que el número depende de qué observaciones quedan en prueba. Elegir método es decidir cuánta varianza y sesgo aceptamos en la estimación. En otro problema consideraría: (1) **tamaño del dataset**: con pocos datos, validación cruzada (repetida) para aprovechar todas las observaciones; con millones de filas, un hold-out ya es estable y más barato; (2) **balance de clases**: con clases muy desbalanceadas es indispensable estratificar, o algunos folds casi no tendrían casos de la clase minoritaria; (3) **costo de entrenamiento**: si entrenar tarda horas, no es viable repetir 25 veces y conviene hold-out o pocos folds. También cuidaría datos con estructura (series de tiempo o grupos), donde las particiones deben respetar el orden o los grupos.


---
## Rúbrica

| Criterio | Excelente | Bueno | Insuficiente | No entregado / Incorrecto | Puntos |
|---|---|---|---|---|---|
| **1.1 Hold-out estratificado** | Partición y ajuste correctos; accuracy calculado e interpretado correctamente. | Partición y ajuste correctos, con un error menor (p. ej. un argumento incompleto) que no afecta el resultado final. | Partición o ajuste con errores que alteran el resultado (p. ej. sin estratificar, sin `random_state`). | Celda sin completar o con errores que impiden su ejecución. | 15 |
| **1.2 Interpretación de la estratificación** | Explica correctamente el riesgo de desbalance en la partición y por qué `stratify` lo evita. | Explica la idea general pero de forma incompleta o imprecisa. | Respuesta genérica, sin relación con el desbalance de clases del dataset. | Sin respuesta. | 5 |
| **2.1 StratifiedKFold** | Configuración correcta de `cv` y `cross_validate`; media y desviación estándar reportadas con `ddof=1`. | Correcto, con un error menor que no cambia el resultado (p. ej. `ddof` por defecto). | Configuración incompleta o métrica mal calculada. | Celda sin completar o con errores que impiden su ejecución. | 15 |
| **2.2 Comparación hold-out vs. k-fold** | Explica con claridad por qué una sola partición no es representativa y cómo k-fold promedia sobre varias particiones. | Identifica la idea central pero sin justificarla completamente. | Comparación superficial, sin referirse a la variabilidad entre particiones. | Sin respuesta. | 10 |
| **3.1 RepeatedStratifiedKFold** | Configuración correcta de `cv` y `cross_validate`; media y desviación estándar reportadas sobre las 25 estimaciones. | Correcto, con un error menor que no cambia el resultado. | Configuración incompleta (p. ej. `n_repeats` incorrecto) o métrica mal calculada. | Celda sin completar o con errores que impiden su ejecución. | 15 |
| **3.2 Fuente de variabilidad adicional** | Identifica correctamente que la repetición varía la configuración de folds entre repeticiones, no solo las observaciones dentro de un fold. | Identifica que hay más variabilidad capturada, sin precisar el mecanismo. | Respuesta genérica o incorrecta sobre qué varía entre repeticiones. | Sin respuesta. | 10 |
| **4.1 Tabla comparativa** | Tabla completa y correcta, con `np.nan` o vacío en la desviación estándar de hold-out. | Tabla completa con un valor incorrecto o mal ubicado. | Tabla incompleta o con varios valores incorrectos. | Celda sin completar. | 10 |
| **4.2 Justificación del método elegido** | Justifica la elección con el argumento de sesgo/varianza de la estimación, no solo con el valor numérico de accuracy. | Justifica la elección, pero basándose solo en qué método dio el accuracy más alto. | Elige un método sin justificación relacionada con sesgo o varianza. | Sin respuesta. | 5 |
| **Reflexión final** | Identifica con claridad los factores relevantes (tamaño del dataset, balance de clases, costo) y los conecta con el método de partición o remuestreo adecuado. | Identifica algunos factores relevantes, de forma parcial. | Reflexión genérica, sin relación específica con los factores de decisión vistos en la sesión. | Sin respuesta. | 15 |

**Nota.** La penalización por entrega tardía es del 5% por día calendario, hasta 7 días, después de lo cual la actividad recibe 0 puntos. Esta penalización se aplica sobre el puntaje ya obtenido con la rúbrica, no como parte de ella.
